In [ ]:
import spikeinterface.full as si
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

from pynapple.process.filtering import apply_bandpass_filter
import pynapple as nap
from nolanlab_ephys.utils import get_recording_folders


In [ ]:
def get_recording_folders(cohort_folder, mouse, day):
    # search the cohort folder for the recording folders that match the mouse and day
    recording_folders = []


    #  need to implement this function to return the correct recording folders

In [ ]:
mouse_days = {
    20: [25,20,21,22],
    21: [25,19,20,21],
    22: [36,35,33,34],
    25: [21,19,22,23],
    26: [14,18,11,12],
    27: [21,20,23,24],
    28: [22,21,19,20],
    29: [22,21,19,20]
}

In [ ]:
all_channel_positions = pd.read_csv("/Volumes/cmvm/sbms/groups/CDBS_SIDB_storage/NolanLab/ActiveProjects/Chris/Cohort12/derivatives/labels/anatomy/mouse_day_channel_ids_brain_location.csv")

theta_dict = {}

session = "VR"

for mouse, days in mouse_days.items():
    theta_dict[mouse] = {}
    for day in days:
        
        session_folder = f"/Users/harryclark/Downloads/COHORT12/M{mouse}/D{day}/{session}/"
        beh = nap.load_file(session_folder + f"sub-{mouse}_day-{day}_ses-{session}_beh.nwb")

        mouseday_channel_positions = all_channel_positions.query(f'mouse == {mouse} & day == {day}')
        base_ch_loc = mouseday_channel_positions.iloc[0]['coord_probe_x']
        channel_mask = mouseday_channel_positions['coord_probe_x'] == base_ch_loc
        channel_ids = mouseday_channel_positions[channel_mask]['channel_id'].values

        all_lfp = np.transpose(beh['LFP'].values[:,channel_mask])

        Fs = 60
        longest_time = np.argmax(beh['moving']['end'] - beh['moving']['start'])
        
        # FOR M25 D23: theta_moving_longest = theta.restrict(beh['moving'][35])
        if mouse == 25 and day == 23:
            longest_time = 35
            
        data = []

        for channel_id, trace in zip(channel_ids, all_lfp):

            channel_y_location = mouseday_channel_positions[mouseday_channel_positions['channel_id'] == channel_id]['coord_probe_y'].values[0]

            samples = len(trace)
            t = np.arange(0,samples)/Fs
            theta = nap.Tsd(t=t, d=trace)
            theta_moving_longest = theta.restrict(beh['moving'][longest_time])

            psd = nap.compute_power_spectral_density(theta_moving_longest,fs=Fs)

            dt = psd.index[1] - psd.index[0]
            total_theta_power = np.sum(psd[(psd.index > 6) & (psd.index < 12)].values)*dt

            data.append([channel_y_location, total_theta_power])

        theta_dict[mouse][day] = data




In [ ]:
def compute_phase(theta_1, theta_2):

    rdata = np.transpose(np.vstack([theta_1, theta_2]))
    
    scaler = StandardScaler()
    scaler.fit(rdata)
    data = scaler.transform(rdata)

    c = np.cov(np.transpose(data))

    phase = np.arccos(min(c[0,1],1))

    return phase

all_channel_positions = pd.read_csv("/Volumes/cmvm/sbms/groups/CDBS_SIDB_storage/NolanLab/ActiveProjects/Chris/Cohort12/derivatives/labels/anatomy/mouse_day_channel_ids_brain_location.csv")

phase_dict = {}

session = "VR"

for mouse, days in mouse_days.items():
    phase_dict[mouse] = {}
    for day in days:
        
        session_folder = f"/Users/harryclark/Downloads/COHORT12/M{mouse}/D{day}/{session}/"
        beh = nap.load_file(session_folder + f"sub-{mouse}_day-{day}_ses-{session}_beh.nwb")

        mouseday_channel_positions = all_channel_positions.query(f'mouse == {mouse} & day == {day}')
        base_ch_loc = mouseday_channel_positions.iloc[0]['coord_probe_x']
        channel_mask = mouseday_channel_positions['coord_probe_x'] == base_ch_loc
        channel_ids = mouseday_channel_positions[channel_mask]['channel_id'].values

        all_lfp = np.transpose(beh['LFP'].values[:,channel_mask])

        Fs = 60
        longest_time = np.argmax(beh['moving']['end'] - beh['moving']['start'])
        
        # FOR M25 D23: theta_moving_longest = theta.restrict(beh['moving'][35])
        if day == 23 and mouse == 25:
            longest_time = 35
            
        data = []

        freq_min = 7.5
        freq_max = 8.5

        trace_base = all_lfp[150]
        samples_base = len(trace_base)
        t_base = np.arange(0,samples_base)/Fs
        theta_base = nap.Tsd(t=t_base, d=trace_base)
        theta_base_moving_longest = theta_base.restrict(beh['moving'][longest_time])
        filtered_theta_base = apply_bandpass_filter(theta_base_moving_longest, cutoff=(freq_min,freq_max))
        
        for channel_id, trace in zip(channel_ids, all_lfp):

            channel_y_location = mouseday_channel_positions[mouseday_channel_positions['channel_id'] == channel_id]['coord_probe_y'].values[0]

            samples = len(trace)
            t = np.arange(0,samples)/Fs
            theta = nap.Tsd(t=t, d=trace)
            theta_moving_longest = theta.restrict(beh['moving'][longest_time])
            filtered_theta = apply_bandpass_filter(theta_moving_longest, cutoff=(freq_min,freq_max))

            phase = compute_phase(filtered_theta_base.values, filtered_theta.values)

            data.append([channel_y_location, phase])

        phase_dict[mouse][day] = data

In [ ]:
gc_dict_1 = {}
gc_dict_2 = {}

for mouse, days in mouse_days.items():
    gc_dict_1[mouse] = {}
    gc_dict_2[mouse] = {}
    for day in days:

        session_folder_1 = f"/Users/harryclark/Downloads/COHORT12/M{mouse}/D{day}/OF1/"
        grid_path_1 = session_folder_1 + "tuning_scores/grid_score.parquet"
        spikes_path_1 = session_folder_1 + f"sub-{mouse}_day-{day}_ses-OF1_srt-kilosort4_clusters.npz"

        clusters_1 = nap.load_file(spikes_path_1)
        grid_scores_1 = pd.read_parquet(grid_path_1)

        grid_cells_1 = grid_scores_1.query('sig == True')

        session_folder_2 = f"/home/nolanlab/Work/Harry_Project/Wolf/M{mouse}/D{day}/OF2/"
        grid_path_2 = session_folder_2 + "tuning_scores/grid_score.parquet"
        spikes_path_2 = session_folder_2 + f"sub-{mouse}_day-{day}_ses-OF2_srt-kilosort4_clusters.npz"

        clusters_2 = nap.load_file(spikes_path_2)
        grid_scores_2 = pd.read_parquet(grid_path_2)

        grid_cells_1 = grid_scores_1.query('sig == True')
        grid_cells_2 = grid_scores_2.query('sig == True')
       
        gc_dict_1[mouse][day] = clusters_1[grid_cells_1['cluster_id'].values]['coord_probe_y']
        gc_dict_2[mouse][day] = clusters_2[grid_cells_2['cluster_id'].values]['coord_probe_y']


In [ ]:
clusters_dict = {}
for mouse, days in mouse_days.items():
    clusters_dict[mouse] = {}
    for day in days:
        session_folder = f"/home/nolanlab/Work/Harry_Project/Wolf/M{mouse}/D{day}/VR/"
        clusters = nap.load_file(session_folder + f"sub-{mouse}_day-{day}_ses-VR_srt-kilosort4_clusters.npz")
        good_clusters = clusters[(clusters['isi_violations_ratio'] < 0.5) & (clusters['presence_ratio'] > 0.9) & (clusters['firing_rate'] > 0.5) & (clusters['snr'] > 1)]
        clusters_dict[mouse][day] = good_clusters['coord_probe_y'].values

In [ ]:
mouse = 20
session_folder = f"/home/nolanlab/Work/Harry_Project/Wolf/M{mouse}/D{day}/VR/"
day = 25
clusters = nap.load_file(session_folder + f"sub-{mouse}_day-{day}_ses-VR_srt-kilosort4_clusters.npz")

In [ ]:
firing_dict = {}
for mouse, days in mouse_days.items():
    firing_dict[mouse] = {}
    for day in days:
        session_folder = f"/home/nolanlab/Work/Harry_Project/Wolf/M{mouse}/D{day}/VR/"
        clusters = nap.load_file(session_folder + f"sub-{mouse}_day-{day}_ses-VR_srt-kilosort4_clusters.npz")
        good_clusters = clusters[(clusters['isi_violations_ratio'] < 0.5) & (clusters['presence_ratio'] > 0.9) & (clusters['firing_rate'] > 0.5) & (clusters['snr'] > 1)]
        firing_dict[mouse][day] = [good_clusters['coord_probe_y'].values, good_clusters['firing_rate'].values]

In [ ]:

cohort_folder = "/Volumes/cmvm/sbms/groups/CDBS_SIDB_storage/NolanLab/ActiveProjects/Harry/EphysNeuropixelData"

noise_dict = {}
locs_dict = {}

si.set_global_job_kwargs(n_jobs=8)

for mouse, days in mouse_days.items():
    noise_dict[mouse] = {}
    locs_dict[mouse] = {}
    for day in days:

        recording_paths = get_recording_folders(cohort_folder, mouse, day)
        print(recording_paths)
        if mouse > 21:
            rec = si.read_openephys(recording_paths[0], stream_id = '0')
        else:
            rec = si.read_zarr(recording_paths[0] + "/recording.zarr")
        left_channels = rec.get_channel_locations()[:,0] == rec.get_channel_locations()[0,0]

        y_locs = rec.get_channel_locations()[:,1][left_channels]
        locs_dict[mouse][day] = y_locs
        noise_dict[mouse][day] = si.get_noise_levels(rec)[left_channels]




In [ ]:
cohort_folder = "/Volumes/cmvm/sbms/groups/CDBS_SIDB_storage/NolanLab/ActiveProjects/Harry/EphysNeuropixelData"

gamma_power_dict = {}
gamma_phase_dict = {}

si.set_global_job_kwargs(n_jobs=8)

for mouse, days in mouse_days.items():
    gamma_power_dict[mouse] = {}
    gamma_phase_dict[mouse] = {}

    for day in days:
        recording_paths = get_recording_folders(cohort_folder, mouse, day)
        vr_mask = ['VR' in recording_path for recording_path in recording_paths]
        vr_rec_path = np.array(recording_paths)[vr_mask][0]
        if mouse > 21:
            rec = si.read_openephys(vr_rec_path, stream_id = '0')
        else:
            rec = si.read_zarr(vr_rec_path + "/recording.zarr")

        session_folder = f"/Users/harryclark/Downloads/COHORT12/M{mouse}/D{day}/{session}/"
        beh = nap.load_file(session_folder + f"sub-{mouse}_day-{day}_ses-{session}_beh.nwb")

        mouseday_channel_positions = all_channel_positions.query(f'mouse == {mouse} & day == {day}')
        base_ch_loc = mouseday_channel_positions.iloc[0]['coord_probe_x']
        channel_mask = mouseday_channel_positions['coord_probe_x'] == base_ch_loc
        channel_ids = mouseday_channel_positions[channel_mask]['channel_id'].values

        Fs = 1000
        longest_time = np.argmax(beh['moving']['end'] - beh['moving']['start'])
        if mouse == 25 and day == 23:
            longest_time = 35
        longest_times = beh['moving'][longest_time]

        start_time = longest_times['start'][0]
        end_time = longest_times['end'][0]

        lfp_rec = si.resample(si.bandpass_filter(rec, freq_min= 20, freq_max=240), Fs)
        sliced_rec = lfp_rec.time_slice(start_time=start_time, end_time=end_time)

        further_filtered_gamma = si.bandpass_filter(sliced_rec, freq_max=55, freq_min=45)

        all_traces = sliced_rec.get_traces(channel_ids=channel_ids)

        data=[]
        phase_data = []

        base_gamma_phase = np.transpose(further_filtered_gamma.get_traces(channel_ids=[f'{channel_ids[0]}']))
                        
        for channel_id, trace in zip(channel_ids, np.transpose(all_traces)):

            channel_y_location = mouseday_channel_positions[mouseday_channel_positions['channel_id'] == channel_id]['coord_probe_y'].values[0]

            samples = len(trace)
            t = np.arange(0,samples)/Fs
            gamma = nap.Tsd(t=t, d=trace)

            psd = nap.compute_power_spectral_density(gamma,fs=Fs)

            dt = psd.index[1] - psd.index[0]
            total_gamma_power = np.sum(psd[(psd.index > 40) & (psd.index < 120)].values)*dt

            data.append([channel_y_location, total_gamma_power])

            gamma_for_phase = np.transpose(further_filtered_gamma.get_traces(channel_ids=[channel_id]))

            gamma_phase = compute_phase(base_gamma_phase, gamma_for_phase)

            phase_data.append([channel_y_location, gamma_phase])

        gamma_power_dict[mouse][day] = data
        gamma_phase_dict[mouse][day] = phase_data


In [ ]:
labels_folder = "/run/user/1000/gvfs/smb-share:server=cmvm.datastore.ed.ac.uk,share=cmvm/sbms/groups/CDBS_SIDB_storage/NolanLab/ActiveProjects/Chris/Cohort12/derivatives/labels/anatomy/"
brain_labels = pd.read_csv(labels_folder + "mouse_day_channel_ids_brain_location_new.csv")

labels_dict = {}

region_colors = {
    'ENT': 'C0',
    'VIS': 'C1',
    'PAR': 'C2',
    'PRE': 'C3',
    'HPF': 'C4',
    'root': 'black',
    'other': 'grey',
}

for mouse, days in mouse_days.items():
    labels_dict[mouse] = {}
    for day in days:
        
        this_mouseday = brain_labels.query(f'mouse == {mouse} & day == {day}')

        points = []

        for contact_id, row in this_mouseday.iterrows():

            brain_region = row['brain_region']
            if 'ENT' in brain_region:
                br = 'ENT'
            elif 'VIS' in brain_region:
                br = 'VIS'
            elif 'root' in brain_region:
                br = 'root'
            elif 'HPF' in brain_region:
                br = 'HPF'
            elif 'PAR' in brain_region:
                br = 'PAR'
            elif 'PRE' in brain_region:
                br = 'PRE'
            else:
                br = 'other'

            points.append([row['coord_probe_y'], br])

        labels_dict[mouse][day] = points



In [ ]:
def remove_outliers(theta, min_q=0.05, max_q=0.95):

    locs = np.array(theta)[:,0]
    values = np.array(theta)[:,1]
    pd_values = pd.Series(values)
    pd_outliers = pd_values.between(pd_values.quantile(min_q), pd_values.quantile(max_q))
    outlier_mask = pd_outliers.values
    without_outliers = np.vstack([locs[outlier_mask], values[outlier_mask]])

    return np.transpose(without_outliers)


In [ ]:

rows = ["noise", "theta", "theta phase", "gamma", "gamma phase", "units", "grid cells"]

import matplotlib.pyplot as plt

mouse_days = {20: [25, 20, 21, 22],
 21: [25, 19, 20, 21],
 22: [36, 35, 33, 34],
 25: [21, 19, 22, 23],
 26: [14, 18, 11, 12],
 27: [21, 20, 23, 24],
 28: [22, 21, 19, 20],
 29: [22, 21, 19, 20]}

for mouse in mouse_days:
    fig, axes = plt.subplots(4,9, figsize=(4*len(rows),30), sharex='col')
    for ax, noise, locs, theta, theta_phase, gamma, gamma_phase, cluster_locations, gc_locs_1, gc_locs_2, day, firing, labels in zip(
        axes, 
        noise_dict[mouse].values(), 
        locs_dict[mouse].values(), 
        theta_dict[mouse].values(), 
        phase_dict[mouse].values(), 
        gamma_power_dict[mouse].values(), 
        gamma_phase_dict[mouse].values(), 
        clusters_dict[mouse].values(), 
        gc_dict_1[mouse].values(), 
        gc_dict_2[mouse].values(),
        gc_dict_2[mouse],
        firing_dict[mouse].values(),
        labels_dict[mouse].values(),
    ):

        # noise plots
        ax[0].scatter(noise, locs)
        ax[0].set_title(f"M{mouse} D{day} --  noise")
        ax[0].grid(axis='y')

        #if day != list(mouse_days.keys())[0]
        # theta plots
        theta_o = remove_outliers(theta)
        ax[1].scatter(np.array(theta_o)[:,1], np.array(theta_o)[:,0])
        ax[1].set_title("theta power")
        ax[1].grid(axis='y')
        
        # theta phase
        theta_phase_o = remove_outliers(theta_phase)
        ax[2].scatter(np.array(theta_phase_o)[:,1], np.array(theta_phase_o)[:,0])
        ax[2].set_title("rel. theta phase")
        ax[2].grid(axis='y')

        # gamma power
        gamma_o = remove_outliers(gamma)
        ax[3].scatter(np.array(gamma_o)[:,1], np.array(gamma_o)[:,0])
        ax[3].set_title("gamma power")
        ax[3].grid(axis='y')

        # gamma phase
        gamma_phase_o = remove_outliers(gamma_phase)
        ax[4].scatter(np.array(gamma_phase_o)[:,1], np.array(gamma_phase_o)[:,0])
        ax[4].set_title("rel. gamma phase")
        ax[4].grid(axis='y')

        # unit density
        ax[5].scatter(-3 + 0*cluster_locations, cluster_locations, c='#0f0f0f20')
        ax[5].hist(cluster_locations, bins=20, orientation='horizontal')
        ax[5].set_title("cluster location")
        ax[5].set_ylim(-150,3010)
        ax[5].grid(axis='y')

        # grid cells
        ax[6].scatter(-1 + 0*gc_locs_1, gc_locs_1, marker='x')
        ax[6].scatter( 1 + 0*gc_locs_2, gc_locs_2, marker='x')
        ax[6].set_ylim(-150,3010)
        ax[6].grid(axis='y')
        ax[6].set_title("grid cell location (OF1, then 2)")
        ax[6].set_xlim(-10,10)

        # firing rate
        ax[7].scatter(0*firing[0], firing[0], s=firing[1]*4, alpha=0.2)
        ax[7].set_ylim(-150,3010)
        ax[7].grid(axis='y')
        ax[7].set_title("Firing rate")

        # harry labels
        labels_np = np.array(labels)
        for region in ['ENT', 'VIS', 'PAR', 'PRE', 'HPF', 'root', 'other']:
            region_labels = labels_np[labels_np[:,1] == region]
            ax[8].scatter(0*(region_labels[:,0].astype('float')), region_labels[:,0].astype('float'), label=region, c=region_colors[region])
        ax[8].legend()
        ax[8].set_ylim(-150,3010)
        ax[8].grid(axis='y')
        ax[8].set_title("Anatomy labels")


    fig.tight_layout()
    fig.savefig(f"M{mouse}_anatomy_helper.pdf")




In [ ]:
# save the computationally expensive stuff

noise_df = pd.DataFrame()
for mouse, days in mouse_days.items():
    for day in days:

        one_day_df = pd.DataFrame()
        one_day_df["y_loc"] = np.array(locs_dict[mouse][day])
        one_day_df["noise_level"] = np.array(noise_dict[mouse][day])
        one_day_df["day"] = day
        one_day_df["mouse"] = mouse

        noise_df = pd.concat([noise_df, one_day_df], ignore_index=True)
    
noise_df.to_csv("noise.csv")

gamma_power_df = pd.DataFrame()
for mouse, days in mouse_days.items():
    for day in days:

        one_day_df = pd.DataFrame()
        one_day_df["y_loc"] = np.array(gamma_power_dict[mouse][day])[:,0]
        one_day_df["gamma_power"] = np.array(gamma_power_dict[mouse][day])[:,1]
        one_day_df["day"] = day
        one_day_df["mouse"] = mouse

        gamma_power_df = pd.concat([gamma_power_df, one_day_df], ignore_index=True)
    
gamma_power_df.to_csv("gamma_power.csv")


theta_power_df = pd.DataFrame()
for mouse, days in mouse_days.items():
    for day in days:

        one_day_df = pd.DataFrame()
        one_day_df["y_loc"] = np.array(theta_dict[mouse][day])[:,0]
        one_day_df["theta_power"] = np.array(theta_dict[mouse][day])[:,1]
        one_day_df["day"] = day
        one_day_df["mouse"] = mouse

        theta_power_df = pd.concat([theta_power_df, one_day_df], ignore_index=True)
    
theta_power_df.to_csv("theta_power.csv")
theta_phase_df = pd.DataFrame()
for mouse, days in mouse_days.items():
    for day in days:

        one_day_df = pd.DataFrame()
        one_day_df["y_loc"] = np.array(phase_dict[mouse][day])[:,0]
        one_day_df["theta_phase"] = np.array(phase_dict[mouse][day])[:,1]
        one_day_df["day"] = day
        one_day_df["mouse"] = mouse

        theta_phase_df = pd.concat([theta_phase_df, one_day_df], ignore_index=True)
    
theta_phase_df.to_csv("theta_phase.csv")
gamma_phase_df = pd.DataFrame()
for mouse, days in mouse_days.items():
    for day in days:

        one_day_df = pd.DataFrame()
        one_day_df["y_loc"] = np.array(gamma_phase_dict[mouse][day])[:,0]
        one_day_df["gamma_phase"] = np.array(gamma_phase_dict[mouse][day])[:,1]
        one_day_df["day"] = day
        one_day_df["mouse"] = mouse

        gamma_phase_df = pd.concat([gamma_phase_df, one_day_df], ignore_index=True)
    
gamma_phase_df.to_csv("gamma_phase.csv")